# Model Context Protocol (MCP)



## browser を用いた LLM 利用

通常の LLM 利用では：

```text id="jlwm9w"
browser
  ↓
文字入力
  ↓
LLM
  ↓
文字出力
  ↓
browser 表示
```

となる。

browser は：

* 入力UI
* 出力UI

として動作している。



## Claude Code / Codex CLI と MCP

一方、Claude Code や Codex CLI では、browser の代わりに CLI application が入出力を担当する。

```text
Claude Code / Codex CLI
  ↓
LLM
  ↓
Claude Code / Codex CLI
```

さらに、LLM が外部 tool を使えるようにすると、次のようになる。

```text
Claude Code / Codex CLI
  ↓
LLM
  ↓
Claude Code / Codex CLI
  ↓
filesystem
git
shell
  ↓
Claude Code / Codex CLI
  ↓
LLM
```

このように、LLM と外部 tool を接続するための protocol が MCP である。

MCP では、この接続を実現するために：

* MCP client
* MCP server

を用いる。

Claude Code や Codex CLI は MCP client を内部に実装しており、tool を呼び出すことができる。

Claude Code や Codex CLI から呼び出される側が
MCP server であり、tool を提供する側である。

```text
Claude Code / Codex CLI
  ↓
MCP client
  ↓
MCP protocolを用いて通信
  ↓
MCP server
  ↓
例）filesystem / git / shell
```

つまり：

```text
MCP client = tool を使う側
MCP server = tool を提供する側
MCP tool   = 実際に呼ばれる機能
```

である。




## MCP とは

[MCP公式仕様](https://modelcontextprotocol.io/introduction?utm_source=chatgpt.com)

MCP（Model Context Protocol）は：

```text id="jlwm9z"
LLM と tool を接続する protocol
```

である。

MCP により：

* filesystem
* GitHub
* database
* Python

などを、
LLM から共通方式で利用できる。



## server と client

一般に：

* server：機能を提供する側
* client：機能を利用する側

である。

Web の例：

```text id="jlwma0"
browser (client)
    ↓
web server
```



## MCP client と MCP server

Claude Code や Codex CLI は、機能を呼び出す側なので：

```text id="jlwma1"
MCP client
```

である。

tool を：

* 発見
* 呼び出し
* 結果取得

する。

一方、MCP server は、

```text id="jlwma2"
機能を提供する側、
```

同時に

```text id="jlwma2"
tool 規格、利用法を公開する側
```

である。

構造は：

```text id="jlwma3"
Claude Code
    ↓
MCP client
    ↓
MCP protocol
    ↓
filesystem server
```

となる。




## LLMとMCP clientとのやり取りの内部表現


LLM では内部的に：

```text
User: ...
System: ...
Assistant: ...
```

のような message 列として会話を扱う。

例えば：

```text
System:
あなたは親切な assistant です。

User:
Python を教えて。

Assistant:
Python は ...
```

のようになる。

現在の ChatGPT や Claude 系では、多くの場合：

```json
[
  {"role": "system", "content": "..."},
  {"role": "user", "content": "..."},
  {"role": "assistant", "content": "..."}
]
```

のような構造化 message 形式で LLM に入出力される。

MCP の tool call も、実際には assistant message の一種として扱われる。

例えば,LLMがtool callを行いたい場合は：

```json
{
  "role": "assistant",
  "tool_call": {
    "name": "show_tree",
    "arguments": {
      "directory": "/home/kino/work",
      "level": 2
    }
  }
}
```

のような形のmessageをLLMがClaude Code/Claude CLI/Code CLIに出力する。（このため、LLMが呼べるtoolを、LLMにまず教える必要がある。）

Claude Code/Claude CLI/Code CLIがtool_call部分をMCP client経由で
に渡し、MCP serverが実行する。

MCP serverの結果をMCP client経由でClaude Code/Claude CLI/Code CLIが

```json
{
  "role": "tool",
  "content": "tree の結果 ..."
}
```

として再び LLM に渡すことにより、MCPの実行・入出力が行われる（LLMが理解する）。

MCPの結果は文字列だが、そのものをClaude Code/Claude CLI/Code CLIが出力するのではなく、LLMが一度受け取って、LLMが内容を整形して出力する。

```text id="jlwma3"
treeツール
    ↓
filesystem server（ツールを含むサーバー）
    ↓
MCP client
    ↓
Claude Code（ツール出力をそのままLLMに送る）
    ↓
　LLM（LLM出力を理解して、整形する）
    ↓
Claude Code（LLM出力を画面に出力するだけ）
```

---



## 作成したfilesystem server の例

以下は：

２つのツール
* directory 構造取得
* file 内容取得

を含んだ MCP server である。

[実行用スクリプト](src/mcp_filesystem_server.py)ではローカル接続を使い、読み取り範囲を起動時の作業ディレクトリ、または環境変数 `MCP_ALLOWED_ROOT` で指定したディレクトリに限定します。

この例は Python MCP SDK 2.x を使います。`8300.LLM/` で次のコマンドを実行します。

```bash
python -m pip install 'mcp>=2.3,<3'
python src/mcp_filesystem_server.py
```

MCP クライアントには Streamable HTTP の接続先として `http://127.0.0.1:8000/mcp` を指定します。

```python
from mcp.server.mcpserver import MCPServer
import subprocess
import os
from pathlib import Path

mcp = MCPServer(name="filesystem-server")

ALLOWED_ROOT = Path(os.environ.get("MCP_ALLOWED_ROOT", Path.cwd())).resolve()


def validate_path(path: str):

    realpath = Path(path).resolve()

    if not realpath.is_relative_to(ALLOWED_ROOT):
        raise ValueError(f"Access denied: {realpath}")

    return realpath


@mcp.tool()
def show_tree(directory: str, level: int = 1) -> str:
    """
    directory を level 階層まで tree 表示する。
    """

    if level < 1:
        return "Error: level must be >= 1"

    if level > 10:
        return "Error: level too large"

    try:
        path = validate_path(directory)

    except Exception as e:
        return str(e)

    result = subprocess.run(
        ["tree", "-L", str(level), path],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        return result.stderr

    return result.stdout


@mcp.tool()
def cat_file(filename: str) -> str:
    """
    file 内容を返す。
    """

    try:
        path = validate_path(filename)

    except Exception as e:
        return str(e)

    if not os.path.isfile(path):
        return f"Not a file: {path}"

    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()

    except Exception as e:
        return str(e)


if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)
```

**@mcp.tool()**によりPythonの関数をMCPで扱えるようにしている。

ツール説明：

`show_tree(directory, level)` は：

```text id="jlwma6"
directory を level 階層まで tree 表示する tool
```

である。

例：

```text id="jlwma7"
show_tree("/home/kino/work", 2)
```

`cat_file(filename)` は：

```text id="jlwma8"
file 内容を返す tool
```

である。

例：

```text id="jlwma9"
cat_file("/home/kino/work/test.py")
```

---


## LLMのツール理解

MCP server 自体は計算機言語を用いたプログラムであり、言語モデルでないので内容を理解していない。

MCP server は：

* tool 名
* docstring
* schema

をLLM client経由でLLMに公開する。

LLM は：

* docstring
* schema

を読んで、

```text id="jlwmbr"
どの tool を使うか
```

を判断する。

例えば：

```python id="jlwmbs"
def show_tree(directory: str, level: int)
```

では、docstring

```text id="jlwmbt"
directory を level 階層まで tree 表示する
```

と、schema

```text id="jlwmbu"
directory: string
level: integer
```

を LLM に渡す。

その結果、

```text id="jlwmbv"
/home/kino/work の directory 構造を２階層まで見せて
```

という自然言語から、show_treeツールを選択し

```text id="jlwmbw"
show_tree("/home/user/work", 2)
```

を生成する。


---



## サーバーとツールの関係



* MCP server：tool をまとめて提供する入れ物
* MCP tool：LLM が実際に呼び出す機能

である。

MCP server は、1つ以上の tool を公開することができる。

```text
MCP server
  ├── tool 1
  ├── tool 2
  └── tool 3
```

例えば、 作成したfilesystem server では：

```text
filesystem-server
  ├── show_tree(directory, level)
  └── cat_file(filename)
```

となる。



Claude Code / Codex CLI から見ると：

```text
Claude Code / Codex CLI
    ↓
MCP client
    ↓
filesystem-server（サーバー名）
    ↓
show_tree / cat_file（サーバーに含まれるツール名）
```

となる。

なお、サーバー名はMCP clientは認識するが、LLMは認識しない。
LLMが認識するのは**ツール名**である。






##  MCPを使った事故

LLM に local PC やクラウドストレージへの file access 権限を与え、コード修正や実行を行わせる system は便利。

一方で、「MCP server や Claude Code が file を削除した」という報告や議論も存在します。

ただし、

```text id="jlwmco"
MCP 自体が危険
```

というより、

```text id="jlwmcp"
LLM に強い filesystem 権限を与えることが危険
```

と考える方が正確です。




## 実際の報告例

Reddit では：

> “I gave you a script that deleted your files”

という投稿があり、Claude が提案した script によりファイルが削除されたと報告されています。 ([Reddit][1])

また、Medium では：

> “Claude Code deleted our production database”

という記事もあります。 ([Medium][2])

ただしこれらは：

* MCP protocol 自体の bug
* Claude が自動的に暴走して全削除

というより、

```text id="jlwm9l"
filesystem / shell / git 権限を持つ Agent に
危険 command を実行させた
```

という問題です。




## Claude repository削除事例

人間も同じミスを犯すので、LLMだけの問題ではない。
(LLMに完全性を求めない方が良い。)

2026年3月31日、Anthropic の **Claude Code** の npm package に、内部 TypeScript source code を含む source map file が誤って同梱されました。Zscaler ThreatLabz は、`@anthropic-ai/claude-code` v2.1.88 に 59.8 MB の `.map` file が含まれ、約 513,000 行・1,906 files の TypeScript code が露出したと説明しています。([Zscaler][1])



## 何が起きたか

流れは次の通りです。

```text
Claude Code release
  ↓
source map file が npm package に混入
  ↓
source code が発見・共有される
  ↓
GitHub に mirror / fork が大量発生
  ↓
Anthropic が DMCA takedown notice を提出
  ↓
意図より広範囲の GitHub repository が takedown 対象になる
```

TechCrunch によれば、Anthropic は漏洩 code を含む repository を削除するため GitHub に DMCA takedown notice を出しましたが、その通知は約 **8,100 repositories** に適用され、正当な fork まで影響を受けたと報じられています。([TechCrunch][2])



## なぜ大量 repository が消えたのか

TechCrunch の報道では、Anthropic 側は「対象 repo が、自社の公開 Claude Code repository とつながる fork network の一部だったため、takedown が意図より広範囲に及んだ」と説明しています。その後、Anthropic は大部分の takedown notice を撤回し、対象を「問題の source code を含む 1 repo と 96 forks」に限定したとされています。([TechCrunch][2])



## Anthropic の説明

Anthropic は、これは security breach ではなく release packaging issue、つまりリリース時の**人為的ミス**だと説明しています。また、customer data や credential は含まれていなかったと述べています。([The Verge][3])



## 重要な教訓

この事例は、LLM agent が直接 repository を削除した事故ではなく、**source leak 対応の takedown 処理が広がりすぎた運用事故**です。

ただし、MCP / Claude Code / coding agent の文脈では重要です。

```text
強い filesystem / git / shell 権限を持つ system は、
意図しない範囲に大きな影響を与えうる
```

という教訓になります。

[1]: https://www.zscaler.com/blogs/security-research/anthropic-claude-code-leak "Anthropic Claude Code Leak | ThreatLabz"
[2]: https://techcrunch.com/2026/04/01/anthropic-took-down-thousands-of-github-repos-trying-to-yank-its-leaked-source-code-a-move-the-company-says-was-an-accident/ "Anthropic took down thousands of GitHub repos trying to yank its leaked source code — a move the company says was an accident | TechCrunch"
[3]: https://www.theverge.com/ai-artificial-intelligence/904776/anthropic-claude-source-code-leak "Claude Code leak exposes a Tamagotchi-style ‘pet’ and an always-on agent | The Verge"





## MCPはなぜ危険か

MCP serverのソースを見ると分かる通り、MCP server は本質的に：

```text id="jlwm9n"
LLM に OS 権限を渡す（ことができる）
```

仕組みです。

例えば：

```python id="jlwm9o"
subprocess.run(...)
```

を使う server は、

実質：

```text id="jlwm9p"
remote shell
```

に近い。

そのため：

* rm -rf
* git reset --hard
* database DROP
* credential leakage

などが起こりうる。


危険なのは：

```python id="jlwm9r"
run_shell(command)
```

のような tool。

例えば、gitコマンド自体も

* commitを取り消していくことで全削除可能

なので単に'git'を呼ぶ実装は危険。

* 'git add', 'git commit'などsubcommandも含めて実装し、'git reset'などは実行できないよう実装する必要がある。




## MCP を安全に使う原則

人間の安全原則とほぼ同じ。

### 1. read-only を基本にする

### 2. write/delete tool を分離

危険例：

```text id="jlwm9s"
delete_file()
run_shell()
```

### 3. ファイルは見れる範囲を限定する。

ソース例だとALLOWED_ROOT を入れる

### 4. production環境を触らせない

研究環境と production を分離。

### 5. human approval

重要操作前に：

```text id="jlwm9t"
confirm required
```

を入れる。


[1]: https://www.reddit.com/r/ClaudeAI/comments/1hkluau/the_worst_mistake_claude_ai_mcp_ever_done/?utm_source=chatgpt.com "The worst mistake Claude AI MCP ever done. : r/ClaudeAI"
[2]: https://medium.com/coding-nexus/the-day-claude-code-deleted-our-production-database-51606d71436e?utm_source=chatgpt.com "The Day Claude Code Deleted Our Production Database"
[3]: https://www.itpro.com/security/a-malicious-mcp-server-is-silently-stealing-user-emails?utm_source=chatgpt.com "A malicious MCP server is silently stealing user emails"
[4]: https://www.techradar.com/pro/security/anthropics-official-git-mcp-server-had-some-worrying-security-flaws-this-is-what-happened-next?utm_source=chatgpt.com "Anthropic's official Git MCP server had some worrying security flaws - this is what happened next"

---


